# Lab 2 | ACF/PACF + đề xuất tham số

**Mục bình chọn:** ACF/PACF + đề xuất tham số — cả 2 dataset

**Hạn bàn giao:** 24/10/2026, 20:00 — bàn giao cấu hình ứng viên cho ARIMA/SARIMA.

> Bạn đọc ACF/PACF để đề xuất các cấu hình đáng thử, kèm lý do. Không chỉ vẽ hai hình rồi chọn số ngẫu nhiên, và chưa cần tự fit toàn bộ mô hình.

**Đối chiếu đề:** Bài cơ bản: Bài 4 trên Dataset 1. Nâng cao: ACF/PACF, tham số thường p/q và mùa vụ P/Q của Dataset 2.

Đây là mô tả nhận việc, không phải lời giải. Các cell code vẫn giữ nguyên `# TODO`.

## Đầu vào, phụ thuộc và việc làm song song

**Cần nhận từ ADF:** code/phép xử lý để tái tạo chuỗi train sau xử lý, d/D đề xuất và kết quả kiểm định.
**Có thể chuẩn bị trước:** import, khung vẽ, cách chọn số lag và phần trình bày.
**Cần chờ để chốt:** kết quả xử lý chuỗi; không kết luận tham số từ test hoặc từ chuỗi chưa phù hợp.

## Cụ thể cần làm gì?

1. Tái tạo chuỗi train theo đúng phép xử lý được bàn giao; xác nhận đang dùng đúng dataset và index.
2. Vẽ ACF/PACF cho cả hai dataset với số lag phù hợp; hình phải có nhãn và khoảng tin cậy.
3. Nhận xét dạng cắt cụt/suy giảm và các lag nổi bật; Dataset 2 chú ý các lag mùa vụ 12, 24, ...
4. Dataset 1: đề xuất 1–2 bộ (p,d,q), trong đó d đối chiếu với phần ADF; viết lý do chọn từng bộ.
5. Dataset 2: đề xuất cấu hình ARIMA baseline và cấu hình SARIMA (p,d,q)(P,D,Q,12).
   Phân biệt tham số thường và mùa vụ, không tự đổi d/D mà chưa trao đổi.
6. Trả lời câu hỏi Bài 4: ACF/PACF thể hiện điều gì, cắt cụt/suy giảm ở đâu và gợi ý mô hình nào.

## Cần bàn giao những gì?

**Gửi cho phần ARIMA và SARIMA:**
- Các đồ thị ACF/PACF có nhận xét, chỉ rõ chuỗi đã xử lý nào được dùng để vẽ.
- 1–2 cấu hình ARIMA Dataset 1; cấu hình ARIMA baseline Dataset 2; cấu hình SARIMA Dataset 2.
- Mỗi cấu hình ghi đủ tham số, dataset, lý do và điều còn chưa chắc chắn.
- Đây là ứng viên để fit/kiểm tra tiếp; ACF/PACF không tự chứng minh mô hình dự báo tốt.

## Quy ước chung

- Chỉ bổ sung code vào các cell `# TODO`; phần nhận xét viết ở cell Markdown ngay bên dưới.
- Đọc CSV chung trong `data/`; không sửa nguồn, không tạo một bản dataset khác.
- Dataset 1 (`a10.csv`) dùng `date`/`value`; Dataset 2 (`airline-passengers.csv`) dùng `Month`/`Passengers`.
- Dùng cùng cách chia **80% train / 20% test theo thứ tự thời gian, không shuffle**:
  Dataset 1 có **163 train / 41 test**; Dataset 2 có **115 train / 29 test**.
- Có thể dùng helper `src/lab2_data.py`, nhưng phần import/load vẫn để bạn tự viết.
  Chỉ lựa chọn phép biến đổi và tham số từ train; không dùng test để chọn mô hình.
- Notebook phải chạy được từ kernel mới; không dựa vào biến đang nằm ở notebook khác.
- Bàn giao qua file notebook/PR có code và nhận xét tái tạo được, không chỉ gửi ảnh chụp.

## Khi nào xem là hoàn thành?

- [ ] Phân tích cả hai dataset và dùng đúng chuỗi train sau xử lý.
- [ ] Có cấu hình thường, mùa vụ và lý do chứ không chỉ có hình.
- [ ] Người fit mô hình nhận được tham số đủ rõ để chạy ngay.

---
## Khu vực thực hiện
Tự bổ sung import, đọc dữ liệu và code vào các cell TODO; ghi kết quả/nhận xét ngay dưới từng phần.

### 1. Tiếp nhận và tái tạo chuỗi đã xử lý

Nhận phép xử lý/d/D từ phần ADF; phân tích chuỗi train đã xử lý.

In [ ]:
import numpy as np
import pandas as pd
from src.lab2_data import load_datasets, chronological_split, SEASON_LENGTH

# Load dữ liệu chuẩn từ dự án
_, _, y1, y2 = load_datasets()
train1, test1 = chronological_split(y1)
train2, test2 = chronological_split(y2)

# Hàm tái tạo chuỗi đã xử lý dựa trên kết quả bàn giao từ phần ADF (d=1, D=1, m=12)
def make_stationary(series, d, D, m=SEASON_LENGTH):
    out = series.copy()
    for _ in range(d):
        out = out.diff(1)
    for _ in range(D):
        out = out.diff(m)
    return out.dropna()

# Áp dụng sai phân cho tập train của cả 2 dataset
stat1 = make_stationary(train1, d=1, D=1, m=SEASON_LENGTH)
stat2 = make_stationary(train2, d=1, D=1, m=SEASON_LENGTH)

print("Shape Dataset 1 sau xử lý:", stat1.shape)
print("Shape Dataset 2 sau xử lý:", stat2.shape)

**Nhận xét:** 
Dataset 1 (a10): Đã tái tạo thành công chuỗi train sau khi áp dụng sai phân bậc 1 (d=1) và sai phân mùa vụ (D=1, m=12) theo đúng kết quả kiểm định ADF trước đó. Chuỗi đã đạt tính dừng để thực hiện ACF/PACF.Dataset 2 (AirPassengers): Đã tái tạo chuỗi train dựa trên phép biến đổi sai phân thường (d=1) và sai phân mùa vụ (D=1, m=12). Quá trình xử lý chỉ áp dụng trên tập train để đảm bảo không bị rò rỉ dữ liệu (data leakage).

### 2. Vẽ và diễn giải ACF/PACF

Vẽ ACF/PACF cả hai dataset; giải thích p/q/P/Q.

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Vẽ ACF và PACF cho Dataset 1
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(stat1, ax=axes[0], lags=24, title="Dataset 1 (a10) - ACF (d=1, D=1)")
plot_pacf(stat1, ax=axes[1], lags=24, title="Dataset 1 (a10) - PACF (d=1, D=1)")
plt.tight_layout()
plt.show()

# Vẽ ACF và PACF cho Dataset 2
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(stat2, ax=axes[0], lags=36, title="Dataset 2 (AirPassengers) - ACF (d=1, D=1)")
plot_pacf(stat2, ax=axes[1], lags=36, title="Dataset 2 (AirPassengers) - PACF (d=1, D=1)")
plt.tight_layout()
plt.show()

**Nhận xét:** Dataset 1 (a10): Biểu đồ PACF giảm nhanh và cắt cụt rõ ở lag 1 (p=1), biểu đồ ACF cắt cụt ở lag 1 (q=1). Cấu hình bậc phi mùa vụ gợi ý cho mô hình là p=1, q=1.Dataset 2 (AirPassengers): Biểu đồ phản ánh rõ đặc trưng mùa vụ mạnh với các điểm dao động tại các độ trễ là bội số của 12 (s=12). Phần phi mùa vụ có dạng tự regressive/moving average cơ bản (gợi ý p=1, q=1 hoặc p=0, q=1).

### 3. Đề xuất cấu hình và bàn giao

Đề xuất cấu hình ARIMA và SARIMA, không dùng test để chọn.

In [ ]:
proposals = {
    "Dataset_1_ARIMA": [
        {"model": "ARIMA(1,1,1)", "p": 1, "d": 1, "q": 1},
        {"model": "ARIMA(2,1,1)", "p": 2, "d": 1, "q": 1}
    ],
    "Dataset_2_SARIMA": [
        {"model": "SARIMA(0,1,1)(0,1,1)_12 [Baseline]", "p": 0, "d": 1, "q": 1, "P": 0, "D": 1, "Q": 1, "s": 12},
        {"model": "SARIMA(1,1,1)(0,1,1)_12", "p": 1, "d": 1, "q": 1, "P": 0, "D": 1, "Q": 1, "s": 12}
    ]
}
import pprint
pprint.pprint(proposals)

**Nhận xét:** Dataset 1 (a10): Đề xuất hai ứng viên mô hình chạy kiểm định AIC/BIC gồm {ARIMA}(1,1,1) và {ARIMA}(2,1,1) (với d=1). Dataset 2 (AirPassengers): Đề xuất cấu hình Baseline kinh điển {SARIMA}(0,1,1)(0,1,1)_{12} và ứng viên thay thế {SARIMA}(1,1,1)(0,1,1)_{12} (với d=1, D=1, s=12).